# GovFriction Lab — independent statistical prediction validation

**Goal:** independently test 30-day recorded nonclosure in NYC 311 administrative records using time-ordered training and testing.

This notebook intentionally **does not import the web app's JavaScript engine**. It uses scikit-learn LogisticRegression and CategoricalNB against a historical-rate baseline. Differences from the browser's specific gradient optimizer and Naive Bayes class-prior smoothing are expected; compare design and out-of-time discrimination/probability metrics, not exact coefficient identity.

The records may contain real administrative data. Do not upload personal/sensitive information publicly. The original NYC source does not verify actual service resolution. This pilot uses only limited non-personal columns.

**GPU not needed.** First upload the SAME frozen CSV used in GovFriction Lab into Kaggle's notebook inputs.

In [ ]:
import csv, json, hashlib, math
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             roc_auc_score, brier_score_loss, log_loss, confusion_matrix)

# If needed, set SOURCE directly to your uploaded, frozen CSV.
SOURCE = None
required = {"unique_key","created_date","closed_date","agency","complaint_type","borough"}
if SOURCE is None:
    candidates = sorted(Path("/kaggle/input").rglob("*.csv")) if Path("/kaggle/input").exists() else sorted(Path("data").glob("*.csv"))
    candidates = [p for p in candidates if required.issubset(set(pd.read_csv(p,nrows=0).columns))]
    if len(candidates)!=1:
        raise RuntimeError(f"Select one explicit SOURCE CSV with required NYC columns; candidates={candidates}")
    SOURCE = candidates[0]
SOURCE = Path(SOURCE)
raw_bytes = SOURCE.read_bytes()
sha256 = hashlib.sha256(raw_bytes).hexdigest()
data = pd.read_csv(SOURCE, dtype=str, keep_default_na=False, encoding="utf-8-sig")
missing = required - set(data.columns)
if missing: raise RuntimeError(f"Source missing fields: {missing}")
print("Frozen input",SOURCE,"rows",len(data),"SHA-256",sha256)

In [ ]:
# Independently reconstruct the cohort and full-follow-up binary target.
START=pd.Timestamp("2025-01-01")
END=pd.Timestamp("2025-04-01")
SNAPSHOT=pd.Timestamp("2026-11-19")
HORIZON=30
SERVICES={"Graffiti","Missed Collection (All Materials)"}
BOROUGHS=("BRONX","BROOKLYN","MANHATTAN","QUEENS","STATEN ISLAND")
counts={x:0 for x in ("input","invalidCreated","duplicate","scopeExcluded","invalidLocationService","invalidClosed","immature","eligible")}
counts["input"]=len(data)
seen=set(); eligible=[]
def parse_stamp(text):
    text=str(text).strip()
    if not text:return None
    try:
        value=pd.Timestamp(text)
        return value if not pd.isna(value) and value.tzinfo is None else None
    except (ValueError, TypeError):return None
for row in data.itertuples(index=False):
    r=row._asdict()
    key=str(r["unique_key"]).strip()
    created=parse_stamp(r["created_date"])
    if not key or created is None:
        counts["invalidCreated"]+=1
        continue
    if key in seen:
        counts["duplicate"]+=1
        continue
    seen.add(key)
    agency=str(r["agency"]).strip().upper()
    service=str(r["complaint_type"]).strip()
    borough=str(r["borough"]).strip().upper()
    if not(START<=created<END and agency=="DSNY" and service in SERVICES):
        counts["scopeExcluded"]+=1
        continue
    if borough not in BOROUGHS:
        counts["invalidLocationService"]+=1
        continue
    raw_closed=str(r["closed_date"]).strip()
    closed=parse_stamp(raw_closed) if raw_closed else None
    if (raw_closed and closed is None) or (closed is not None and closed<created):
        counts["invalidClosed"]+=1
        continue
    if created>SNAPSHOT:
        counts["scopeExcluded"]+=1
        continue
    if SNAPSHOT-created<pd.Timedelta(days=HORIZON):
        counts["immature"]+=1
        continue
    late=(closed is None or closed>SNAPSHOT or closed-created>pd.Timedelta(days=HORIZON))
    eligible.append({"created":created,"service":service,"borough":borough,
                     "dow":(created.weekday()+1)%7,"hour":created.hour,"label":int(late)})
counts["eligible"]=len(eligible)
print("Audit",counts)
rows=pd.DataFrame(eligible)
if rows.empty:raise RuntimeError("No eligible source records. Check cohort and dataset.")

In [ ]:
# Match the app's temporal split and prediction-time features, but fit models independently.
window_days=int((END-START).total_seconds()//86400)
holdout_start=START+pd.Timedelta(days=int(math.floor(window_days*2/3)))
train_end=holdout_start-pd.Timedelta(days=HORIZON)
if train_end<=START or END-holdout_start<pd.Timedelta(days=7):
    raise RuntimeError("Study window cannot support a mature training group and later holdout.")
train=rows.loc[rows.created<train_end].copy()
test=rows.loc[rows.created>=holdout_start].copy()
gap=len(rows)-len(train)-len(test)
if len(train)<40 or len(test)<20 or min(train.label.sum(),len(train)-train.label.sum())<3 or min(test.label.sum(),len(test)-test.label.sum())<2:
    raise RuntimeError("Insufficient time-separated examples of both outcomes. Do not report model performance.")
def logistic_features(frame):
    arr=[]
    for r in frame.itertuples(index=False):
        arr.append([int(r.service=="Graffiti")]+[int(r.borough==v) for v in BOROUGHS[1:]]+
                   [np.sin(2*np.pi*r.dow/7),np.cos(2*np.pi*r.dow/7),
                    np.sin(2*np.pi*r.hour/24),np.cos(2*np.pi*r.hour/24)])
    return np.array(arr,dtype=float)
def categorical_features(frame):
    return np.array([[int(r.service=="Graffiti"),BOROUGHS.index(r.borough),
                      int(r.dow),int(r.hour//6)] for r in frame.itertuples(index=False)],dtype=int)
y_train=train.label.to_numpy(dtype=int);y_test=test.label.to_numpy(dtype=int)
historical=np.repeat(y_train.mean(),len(test))
logistic=LogisticRegression(solver="lbfgs",max_iter=2000,C=1/(0.03*len(train)),tol=1e-9)
logistic.fit(logistic_features(train),y_train)
p_logistic=logistic.predict_proba(logistic_features(test))[:,1]
bayes=CategoricalNB(alpha=1,min_categories=[2,5,7,4])
bayes.fit(categorical_features(train),y_train)
p_bayes=bayes.predict_proba(categorical_features(test))[:,1]
print("Train:",len(train),"; gap:",gap,"; holdout:",len(test))
print("Train ends (exclusive)",train_end.date(),"; holdout starts",holdout_start.date())
print("Training nonclosure",y_train.mean(),"; holdout nonclosure",y_test.mean())

In [ ]:
# Fixed 50% threshold. For calibration, compare observed and predicted frequencies in 20-point bins.
def report_line(name, p):
    predicted=(p>=.5).astype(int)
    tn,fp,fn,tp=confusion_matrix(y_test,predicted,labels=[0,1]).ravel()
    return {"model":name,"source_sha256":sha256,"train_end_exclusive":str(train_end.date()),
            "holdout_start":str(holdout_start.date()),"training_n":len(train),"holdout_n":len(test),
            "holdout_event_rate":float(y_test.mean()),
            "accuracy":float(accuracy_score(y_test,predicted)),
            "balanced_accuracy":float(balanced_accuracy_score(y_test,predicted)),
            "precision":float(precision_score(y_test,predicted,zero_division=0)),
            "recall":float(recall_score(y_test,predicted,zero_division=0)),
            "roc_auc":float(roc_auc_score(y_test,p)),
            "brier":float(brier_score_loss(y_test,p)),
            "log_loss":float(log_loss(y_test,p,labels=[0,1])),
            "tp":int(tp),"tn":int(tn),"fp":int(fp),"fn":int(fn)}
predictions={"Historical-rate baseline":historical,
             "L2-regularized logistic regression":p_logistic,
             "Categorical Naive Bayes":p_bayes}
summary=pd.DataFrame([report_line(name,p) for name,p in predictions.items()])
display(summary[["model","holdout_n","accuracy","balanced_accuracy","recall","roc_auc","brier","log_loss"]])
summary.to_csv("prediction_kaggle_metrics.csv",index=False)
bins=[]
for name,p in predictions.items():
    index=np.minimum(4,np.floor(p*5).astype(int))
    for bucket in range(5):
        m=(index==bucket)
        bins.append({"model":name,"bin":bucket+1,"n":int(m.sum()),
                     "predicted":float(p[m].mean()) if m.any() else None,
                     "observed":float(y_test[m].mean()) if m.any() else None})
pd.DataFrame(bins).to_csv("prediction_kaggle_calibration.csv",index=False)
manifest={"sha256":sha256,"rows_loaded":len(data),"audit":counts,
          "split":{"trainEndExclusive":str(train_end.date()),"holdoutStart":str(holdout_start.date()),
                   "trainingN":len(train),"holdoutN":len(test),"embargoN":gap},
          "methods":"scikit-learn LogisticRegression and CategoricalNB: independent estimators",
          "warning":"Prototype performance only; comparison requires exact input checksum and cohort."}
Path("prediction_kaggle_manifest.json").write_text(json.dumps(manifest,indent=2))
print("Exported prediction_kaggle_metrics.csv, prediction_kaggle_calibration.csv and prediction_kaggle_manifest.json")